In [ ]:
import torch
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, roc_auc_score

In [ ]:
class TabularDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [ ]:
from torch.utils.data import Dataset

class TabularDataset(Dataset):
    def __init__(self, X, y=None):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = self.X[idx]

        if self.y is not None:
            return x, self.y[idx]
        else:
            return x

In [ ]:
from torch.utils.data import Dataset
import torch

class TabularDataset(Dataset):
    def __init__(self, X, y=None):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long) if y is not None else None

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        if self.y is not None:
            return self.X[idx], self.y[idx]
        else:
            return self.X[idx]

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, in_features, out_features, dropout=0.3):
        super().__init__()

        self.fc1 = nn.Linear(in_features, out_features)
        self.bn1 = nn.BatchNorm1d(out_features)

        self.fc2 = nn.Linear(out_features, out_features)
        self.bn2 = nn.BatchNorm1d(out_features)

        self.dropout = nn.Dropout(dropout)

        # Projection if dimensions differ
        self.shortcut = nn.Sequential()
        if in_features != out_features:
            self.shortcut = nn.Sequential(
                nn.Linear(in_features, out_features),
                nn.BatchNorm1d(out_features)
            )

    def forward(self, x):
        identity = self.shortcut(x)

        out = self.fc1(x)
        out = self.bn1(out)
        out = F.relu(out)
        out = self.dropout(out)

        out = self.fc2(out)
        out = self.bn2(out)

        out += identity
        out = F.relu(out)

        return out

In [ ]:
class WideResNetTabular(nn.Module):
    def __init__(self, input_dim, num_classes, width=256, depth=4, dropout=0.3):
        super().__init__()

        self.input_layer = nn.Sequential(
            nn.Linear(input_dim, width),
            nn.BatchNorm1d(width),
            nn.ReLU()
        )

        # Stack residual blocks
        self.blocks = nn.Sequential(
            *[ResidualBlock(width, width, dropout) for _ in range(depth)]
        )

        self.head = nn.Sequential(
            nn.Linear(width, width // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(width // 2, num_classes)
        )

    def forward(self, x):
        x = self.input_layer(x)
        x = self.blocks(x)
        x = self.head(x)
        return x

In [ ]:
dft = pd.read_csv('/content/drive/MyDrive/CaresAI/tnm_staging/data_v2/bert_encoding/encoded_train_t.csv')
#dfn = pd.read_csv('/content/drive/MyDrive/CaresAI/tnm_staging/data_v2/bert_encoding/encoded_train_n.csv')
#dfm = pd.read_csv('/content/drive/MyDrive/CaresAI/tnm_staging/data_v2/bert_encoding/encoded_train_m.csv')
val = pd.read_csv("/content/drive/MyDrive/CaresAI/tnm_staging/data_v2/bert_encoding/validation_enc.csv")
test = pd.read_csv("/content/drive/MyDrive/CaresAI/tnm_staging/data_v2/tfid_encoded/test_encoded.csv")

In [ ]:
dft.head(3)

,patient_filename,text,text_embedding_clinicalbert,text_embedding_biobert,text_embedding_pubmedbert,tfidf_vectors
0,TNM_0000,Final Surgical Pathology Report. Procedure: Re...,"[0.2552698850631714, -0.34963372349739075, -0....","[0.3232501745223999, -0.29840439558029175, -0....","[-0.1003669872879982, 0.12377656996250153, -0....","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
1,TNM_0001,Laterality: Path Report:Tumor size (in cm): 6....,"[0.16342613101005554, -0.27544984221458435, -0...","[0.20009441673755646, -0.25146111845970154, 0....","[-0.07022151350975037, 0.11711522936820984, -0...","[0.0, 0.14075115515829223, 0.0, 0.0, 0.0, 0.0,..."
2,TNM_0002,Clinical Diagnosis & History: T1 adenocarcinom...,"[0.18401573598384857, -0.27159935235977173, -0...","[0.22688910365104675, -0.26918330788612366, -0...","[-0.07323721051216125, 0.11752274632453918, -0...","[0.0, 0.0, 0.0, 0.051518435189986396, 0.044091..."


In [ ]:
yt = dft['t']
Xt = dft[['text_embedding_clinicalbert', 'text_embedding_biobert', 'text_embedding_pubmedbert']]
X_train, X_test, y_train, y_test = train_test_split(Xt, yt, test_size=0.20, stratify=yt)

In [ ]:
#y_train = dft['t']
#X_train = dft[['text_embedding_clinicalbert', 'text_embedding_biobert', 'text_embedding_pubmedbert']]
#y_val = val['t']
val = val[['text_embedding_clinicalbert', 'text_embedding_biobert', 'text_embedding_pubmedbert']]
test = test[['text_embedding_clinicalbert', 'text_embedding_biobert', 'text_embedding_pubmedbert']]

In [ ]:
import ast

def parse_list_string(list_string):
    return ast.literal_eval(list_string)

X_train_list = []

for index, row in X_train.iterrows():
    combined_vector = np.concatenate([
        parse_list_string(row['text_embedding_clinicalbert']),
        parse_list_string(row['text_embedding_biobert']),
        parse_list_string(row['text_embedding_pubmedbert'])
    ])
    X_train_list.append(combined_vector)

X_train_t = np.array(X_train_list)
y_train_t = y_train.values

print(f"Shape of X_train: {X_train_t.shape}")
print(f"Shape of y_train: {y_train_t.shape}")

Shape of X_train: (4682, 2304)
Shape of y_train: (4682,)


In [ ]:
X_test_list_t = []
# text_embedding_clinicalbert, text_embedding_biobert, text_embedding_pubmedbert
for index, row in X_test.iterrows():
    combined_vector = np.concatenate([
        parse_list_string(row['text_embedding_clinicalbert']),
        parse_list_string(row['text_embedding_biobert']),
        parse_list_string(row['text_embedding_pubmedbert'])
    ])
    X_test_list_t.append(combined_vector)

X_test_t = np.array(X_test_list_t)
y_test_t = y_test.values

print(f"Shape of X_train: {X_test_t.shape}")
print(f"Shape of y_train: {y_test_t.shape}")

Shape of X_train: (1171, 2304)
Shape of y_train: (1171,)


In [ ]:
test_list_t = []
# text_embedding_clinicalbert, text_embedding_biobert, text_embedding_pubmedbert
for index, row in test.iterrows():
    combined_vector = np.concatenate([
        parse_list_string(row['text_embedding_clinicalbert']),
        parse_list_string(row['text_embedding_biobert']),
        parse_list_string(row['text_embedding_pubmedbert'])
    ])
    test_list_t.append(combined_vector)

test_t = np.array(test_list_t)
#y_test_t = y_test.values

print(f"Shape of X_train: {test_t.shape}")
#print(f"Shape of y_train: {y_test_t.shape}")

Shape of X_train: (2599, 2304)


In [ ]:
y_train_t = y_train_t - y_train_t.min()
y_test_t = y_test_t - y_test_t.min()
#print(type(y_train_t[0]), y_train_t[:10])

In [ ]:
train_dataset = TabularDataset(X_train_t, y_train_t)
val_dataset = TabularDataset(X_test_t, y_test_t)
test_dataset = TabularDataset(test_t)

In [ ]:
num_classes = len(np.unique(y_train))
print("Unique classes:", np.unique(y_train))
print("Num classes:", num_classes)

Unique classes: [1. 2. 3. 4.]
Num classes: 4


In [ ]:
print("Unique labels:", set(y_train_t))
print("Num classes:", num_classes)
print("Max label:", max(y_train_t))

Unique labels: {np.float64(0.0), np.float64(1.0), np.float64(2.0), np.float64(3.0)}
Num classes: 4
Max label: 3.0


In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64)

In [ ]:
#from sklearn.preprocessing import StandardScaler

#scaler = StandardScaler()
#X_train = scaler.fit_transform(X_train)
#X_val = scaler.transform(X_val)

In [ ]:
# width=512, 768
model = WideResNetTabular(
    input_dim=X_train_t.shape[1],
    num_classes=num_classes,
    width=128,
    depth=4,
    dropout=0.3
)
# 128, 0.1 = f1-0.6; auc-0.81
# 128, 0.2, F1 Micro: 0.5977, AUC: 0.8173
# 256, 0.2, F1 Micro: 0.5765; AUC: 0.8100;

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

WideResNetTabular(
  (input_layer): Sequential(
    (0): Linear(in_features=2304, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
  )
  (blocks): Sequential(
    (0): ResidualBlock(
      (fc1): Linear(in_features=128, out_features=128, bias=True)
      (bn1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (fc2): Linear(in_features=128, out_features=128, bias=True)
      (bn2): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (dropout): Dropout(p=0.3, inplace=False)
      (shortcut): Sequential()
    )
    (1): ResidualBlock(
      (fc1): Linear(in_features=128, out_features=128, bias=True)
      (bn1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (fc2): Linear(in_features=128, out_features=128, bias=True)
      (bn2): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_run

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)
criterion = torch.nn.CrossEntropyLoss()

In [ ]:
from sklearn.metrics import f1_score, roc_auc_score

class EarlyStopping:
    def __init__(self, patience=5, min_delta=0.0, mode="max"):
        self.patience = patience
        self.min_delta = min_delta
        self.mode = mode  # "max" for metrics like F1/AUC
        self.best_score = None
        self.counter = 0
        self.early_stop = False

    def step(self, score):
        if self.best_score is None:
            self.best_score = score
            return False

        improvement = (score - self.best_score) if self.mode == "max" else (self.best_score - score)

        if improvement > self.min_delta:
            self.best_score = score
            self.counter = 0
        else:
            self.counter += 1

        if self.counter >= self.patience:
            self.early_stop = True

        return self.early_stop

In [ ]:
def evaluate_metrics(model, loader, num_classes):
    model.eval()

    all_preds = []
    all_probs = []
    all_labels = []

    with torch.no_grad():
        for X_batch, y_batch in loader:
            logits = model(X_batch)

            probs = torch.softmax(logits, dim=1)
            preds = torch.argmax(probs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())

    all_preds = np.array(all_preds)
    all_probs = np.array(all_probs)
    all_labels = np.array(all_labels)

    # F1 scores
    f1_macro = f1_score(all_labels, all_preds, average="macro")
    f1_micro = f1_score(all_labels, all_preds, average="micro")
    f1_weighted = f1_score(all_labels, all_preds, average="weighted")

    # AUC (multiclass)
    try:
        auc = roc_auc_score(all_labels, all_probs, multi_class="ovr")
    except ValueError:
        auc = float("nan")  # happens if a class is missing in batch

    return {
        "f1_macro": f1_macro,
        "f1_micro": f1_micro,
        "f1_weighted": f1_weighted,
        "auc": auc
    }

In [ ]:
def train_model(model, train_loader, val_loader, optimizer, criterion, num_classes, epochs=70):
    import torch

    early_stopper = EarlyStopping(patience=7, mode="max")

    best_f1 = 0
    best_auc = 0
    best_score = 0  # combined score (recommended)

    best_metrics = None
    best_epoch = 0

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()
            logits = model(X_batch)
            loss = criterion(logits, y_batch)

            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)

        # Evaluate
        metrics = evaluate_metrics(model, val_loader, num_classes)

        f1 = metrics["f1_macro"]
        auc = metrics["auc"]

        print(f"Epoch {epoch+1}")
        print(f"Loss: {avg_loss:.4f}")
        print(f"F1 Macro: {f1:.4f}")
        print(f"F1 Weighted: {metrics['f1_weighted']:.4f}")
        print(f"AUC: {auc:.4f}")
        print("-" * 40)

        #  Combined score (more stable than OR condition)
        combined_score = 0.5 * f1 + 0.5 * auc

        #  Save best model based on combined score
        if combined_score > best_score:
            best_score = combined_score
            best_f1 = f1
            best_auc = auc
            best_metrics = metrics
            best_epoch = epoch + 1

            torch.save({
                'epoch': epoch + 1,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'f1_macro': f1,
                'auc': auc
            }, "best_model.pt")

            print(f" Saved best model (Epoch {epoch+1}, F1={f1:.4f}, AUC={auc:.4f})")

        # Early stopping (still based on F1)
        if early_stopper.step(f1):
            print("Early stopping triggered.")
            break

    # Final best summary
    print("\n===== BEST MODEL =====")
    print(f"Best Epoch: {best_epoch}")
    print(f"Best F1 Macro: {best_metrics['f1_macro']:.4f}")
    print(f"Best F1 Micro: {best_metrics['f1_micro']:.4f}")
    print(f"Best F1 Weighted: {best_metrics['f1_weighted']:.4f}")
    print(f"Best AUC: {best_metrics['auc']:.4f}")

In [ ]:
def evaluate_metrics(model, loader, num_classes, threshold=None):
    import numpy as np
    from sklearn.metrics import f1_score, roc_auc_score

    model.eval()

    all_preds = []
    all_probs = []
    all_labels = []

    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)

            logits = model(X_batch)
            probs = torch.softmax(logits, dim=1)

            # Store probabilities
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())

    all_probs = np.array(all_probs)
    all_labels = np.array(all_labels)

    # Prediction logic
    if num_classes == 2:
        # Binary case → use threshold if provided
        pos_probs = all_probs[:, 1]

        if threshold is not None:
            all_preds = (pos_probs >= threshold).astype(int)
        else:
            all_preds = (pos_probs >= 0.5).astype(int)
    else:
        # Multiclass → argmax
        all_preds = np.argmax(all_probs, axis=1)

    # F1 scores
    f1_macro = f1_score(all_labels, all_preds, average="macro")
    f1_micro = f1_score(all_labels, all_preds, average="micro")
    f1_weighted = f1_score(all_labels, all_preds, average="weighted")

    # AUC
    try:
        if num_classes == 2:
            auc = roc_auc_score(all_labels, all_probs[:, 1])
        else:
            auc = roc_auc_score(all_labels, all_probs, multi_class="ovr")
    except ValueError:
        auc = float("nan")

    return {
        "f1_macro": f1_macro,
        "f1_micro": f1_micro,
        "f1_weighted": f1_weighted,
        "auc": auc,
        "probs": all_probs,     # needed for threshold tuning
        "labels": all_labels    # needed for threshold tuning
    }

In [ ]:
train_model(
            model, train_loader, val_loader, optimizer,\
            criterion, num_classes=num_classes, epochs=70
            )

Epoch 1
Loss: 1.5101
F1 Macro: 0.2148
F1 Weighted: 0.2098
AUC: 0.5694
----------------------------------------
 Saved best model (Epoch 1, F1=0.2148, AUC=0.5694)
Epoch 2
Loss: 1.4529
F1 Macro: 0.2452
F1 Weighted: 0.2450
AUC: 0.6056
----------------------------------------
 Saved best model (Epoch 2, F1=0.2452, AUC=0.6056)
Epoch 3
Loss: 1.4109
F1 Macro: 0.2678
F1 Weighted: 0.2819
AUC: 0.6243
----------------------------------------
 Saved best model (Epoch 3, F1=0.2678, AUC=0.6243)
Epoch 4
Loss: 1.3697
F1 Macro: 0.3174
F1 Weighted: 0.3431
AUC: 0.6461
----------------------------------------
 Saved best model (Epoch 4, F1=0.3174, AUC=0.6461)
Epoch 5
Loss: 1.3540
F1 Macro: 0.3512
F1 Weighted: 0.3894
AUC: 0.6621
----------------------------------------
 Saved best model (Epoch 5, F1=0.3512, AUC=0.6621)
Epoch 6
Loss: 1.3221
F1 Macro: 0.3479
F1 Weighted: 0.3970
AUC: 0.6727
----------------------------------------
 Saved best model (Epoch 6, F1=0.3479, AUC=0.6727)
Epoch 7
Loss: 1.3159
F1 Macr

In [ ]:
checkpoint = torch.load("/content/best_model.pt", weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

WideResNetTabular(
  (input_layer): Sequential(
    (0): Linear(in_features=2304, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
  )
  (blocks): Sequential(
    (0): ResidualBlock(
      (fc1): Linear(in_features=128, out_features=128, bias=True)
      (bn1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (fc2): Linear(in_features=128, out_features=128, bias=True)
      (bn2): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (dropout): Dropout(p=0.3, inplace=False)
      (shortcut): Sequential()
    )
    (1): ResidualBlock(
      (fc1): Linear(in_features=128, out_features=128, bias=True)
      (bn1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (fc2): Linear(in_features=128, out_features=128, bias=True)
      (bn2): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_run

In [ ]:
# Threshold tuning function
import numpy as np
from sklearn.metrics import f1_score

def find_best_threshold(y_true, probs):
    thresholds = np.linspace(0.1, 0.9, 100)

    best_t = 0.5
    best_f1 = 0

    for t in thresholds:
        preds = (probs >= t).astype(int)
        f1 = f1_score(y_true, preds)

        if f1 > best_f1:
            best_f1 = f1
            best_t = t

    print(f"Best Threshold: {best_t:.3f} | Best F1: {best_f1:.4f}")
    return best_t, best_f1

In [ ]:
def get_model_probs(model, loader):
    import numpy as np
    import torch

    model.eval()
    all_probs = []

    with torch.no_grad():
        for X_batch, _ in loader:  # labels not needed here
            X_batch = X_batch.to(device)

            logits = model(X_batch)
            probs = torch.softmax(logits, dim=1)

            all_probs.extend(probs.cpu().numpy())

    return np.array(all_probs)

In [ ]:
# Get probabilities from model
val_probs = get_model_probs(model, val_loader)  # your inference function

# Tune threshold
best_t, best_f1 = find_best_threshold(val_loader, val_probs)

# Final predictions
final_preds = (val_probs >= best_t).astype(int) # test_dataset,

ValueError: Found input variables with inconsistent numbers of samples: [19, 1171]

In [ ]:
def predict(model, loader, threshold=0.75):
    import numpy as np
    import torch

    model.eval()
    all_probs = []

    with torch.no_grad():
        for batch in loader:
            # Handle both cases
            if isinstance(batch, (list, tuple)):
                X_batch = batch[0]
            else:
                X_batch = batch

            X_batch = X_batch.to(device)

            logits = model(X_batch)
            probs = torch.softmax(logits, dim=1)

            all_probs.extend(probs.cpu().numpy())

    all_probs = np.array(all_probs)

    # Binary classification
    preds = (all_probs[:, 1] >= threshold).astype(int)

    return preds, all_probs

In [ ]:
# Load best model
#checkpoint = torch.load("best_model.pt")
#model.load_state_dict(checkpoint["model_state_dict"])

# Create test loader
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# Predict
preds, probs = predict(model, test_loader, threshold=0.75)

,text_embedding_clinicalbert,text_embedding_biobert,text_embedding_pubmedbert,prediction
0,"[0.2552698850631714, -0.34963372349739075, -0....","[0.3232501745223999, -0.29840439558029175, -0....","[-0.1003669872879982, 0.12377656996250153, -0....",0
1,"[0.16342613101005554, -0.27544984221458435, -0...","[0.20009441673755646, -0.25146111845970154, 0....","[-0.07022151350975037, 0.11711522936820984, -0...",1
2,"[0.18401573598384857, -0.27159935235977173, -0...","[0.22688910365104675, -0.26918330788612366, -0...","[-0.07323721051216125, 0.11752274632453918, -0...",0
3,"[0.27331188321113586, -0.11066050827503204, -0...","[0.1623775064945221, -0.15081897377967834, -0....","[-0.08789176493883133, 0.1208319365978241, -0....",1
4,"[0.218575119972229, -0.3167060613632202, -0.12...","[0.15919098258018494, -0.2981928586959839, -0....","[-0.0969555526971817, 0.1596442610025406, -0.0...",0
